# Task 3: Consultas analíticas e dashboard

Neste notebook, realizamos consultas no **Amazon Athena** sobre os dados transformados na Task 2 e construímos um dashboard interativo.

In [1]:
import awswrangler as wr
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
import os
from dotenv import load_dotenv
from IPython.display import display, clear_output

# Carrega variáveis do rds_connection.env gerado pelo run_etl.py
load_dotenv(dotenv_path="rds_connection.env")

# Configurações Dinâmicas
GLUE_DATABASE = os.getenv("GLUE_DATABASE_NAME", "classicmodels_gold")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP_NAME", "classicmodels_workgroup")

print(f"Usando Database: {GLUE_DATABASE}")
print(f"Usando Workgroup: {ATHENA_WORKGROUP}")

sns.set_theme(style="whitegrid")

Usando Database: classicmodels_gold_v3
Usando Workgroup: classicmodels_workgroup_v3


## 4.2 - Consulta exploratória em dimensão

Inspecionando o catálogo de produtos no modelo analítico.

In [2]:
query_products = f"""
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM {GLUE_DATABASE}.dim_products
LIMIT 20
"""

df_products = wr.athena.read_sql_query(sql=query_products, database=GLUE_DATABASE, workgroup=ATHENA_WORKGROUP)
df_products.head()

,product_id,product_name,product_line,product_vendor
0,S24_1444,1970 Dodge Coronet,Classic Cars,Highway 66 Mini Classics
1,S24_1628,1966 Shelby Cobra 427 S/C,Classic Cars,Carousel DieCast Legends
2,S24_3371,1971 Alpine Renault 1600s,Classic Cars,Welly Diecast Productions
3,S24_1046,1970 Chevy Chevelle SS 454,Classic Cars,Unimax Art Galleries
4,S18_4027,1970 Triumph Spitfire,Classic Cars,Min Lin Diecast


## 4.3 - Vendas totais por país

Calculando o total de vendas por país juntando a fato à dimensão de país.

In [3]:
query_sales_country = f"""
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM {GLUE_DATABASE}.fact_orders
JOIN {GLUE_DATABASE}.dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

df_sales_country = wr.athena.read_sql_query(sql=query_sales_country, database=GLUE_DATABASE, workgroup=ATHENA_WORKGROUP)
df_sales_country

,country,total_sales
0,USA,3273280.05
1,Spain,1099389.09
2,France,1007374.02
3,Australia,562582.59
4,New Zealand,476847.01
5,UK,436947.44
6,Italy,360616.81
7,Finland,295149.35
8,Singapore,263997.78
9,Denmark,218994.92


## 4.4 - Detalhamento por data, linha de produto, produto e país

Obtendo a base analítica completa.

In [4]:
query_detailed = f"""
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM {GLUE_DATABASE}.fact_orders
JOIN {GLUE_DATABASE}.dim_products ON fact_orders.product_id = dim_products.product_id
JOIN {GLUE_DATABASE}.dim_countries ON fact_orders.country_key = dim_countries.country_key
JOIN {GLUE_DATABASE}.dim_dates ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

df_detailed = wr.athena.read_sql_query(sql=query_detailed, database=GLUE_DATABASE, workgroup=ATHENA_WORKGROUP)
df_detailed['full_date'] = pd.to_datetime(df_detailed['full_date'])
df_detailed.head()

,full_date,product_line,product_name,country,total_sales
0,2004-02-12,Classic Cars,1969 Corvair Monza,Ireland,4532.40
1,2004-09-09,Ships,The Mayflower,Italy,2260.55
2,2004-11-17,Vintage Cars,1917 Grand Touring Sedan,USA,6806.80
3,2004-08-02,Classic Cars,1970 Plymouth Hemi Cuda,USA,2577.54
4,2004-11-19,Classic Cars,1958 Chevy Corvette Limited Edition,USA,1085.04


## 4.5 - Dashboard Interativo

In [ ]:
# Widgets de Filtro
countries = sorted(df_detailed['country'].unique().tolist())
product_lines = sorted(df_detailed['product_line'].unique().tolist())

dropdown_country = widgets.Dropdown(options=['Todos'] + countries, value='Todos', description='País:')
dropdown_line = widgets.Dropdown(options=['Todos'] + product_lines, value='Todos', description='Linha:')
slider_top_n = widgets.IntSlider(value=5, min=1, max=20, step=1, description='Top N:')
date_picker_start = widgets.DatePicker(description='Início', value=df_detailed['full_date'].min())
date_picker_end = widgets.DatePicker(description='Fim', value=df_detailed['full_date'].max())

output_plot = widgets.Output()

def update_dashboard(change):
    with output_plot:
        clear_output(wait=True)
        
        # Aplicar filtros
        df_filtered = df_detailed.copy()
        
        # Filtro de Data
        start_date = pd.to_datetime(date_picker_start.value)
        end_date = pd.to_datetime(date_picker_end.value)
        df_filtered = df_filtered[(df_filtered['full_date'] >= start_date) & (df_filtered['full_date'] <= end_date)]
        
        # Filtro de País
        if dropdown_country.value != 'Todos':
            df_filtered = df_filtered[df_filtered['country'] == dropdown_country.value]
            
        # Filtro de Linha de Produto
        if dropdown_line.value != 'Todos':
            df_filtered = df_filtered[df_filtered['product_line'] == dropdown_line.value]
            
        # Agrupar por Produto e pegar Top N
        df_top = df_filtered.groupby('product_name')['total_sales'].sum().reset_index()
        df_top = df_top.sort_values(by='total_sales', ascending=False).head(slider_top_n.value)
        
        # Plot
        plt.figure(figsize=(12, 6))
        sns.barplot(data=df_top, x='total_sales', y='product_name', palette='viridis', hue='product_name', legend=False)
        plt.title(f"Top {slider_top_n.value} Produtos por Vendas")
        plt.xlabel("Vendas Totais")
        plt.ylabel("Produto")
        plt.show()

# Observadores
dropdown_country.observe(update_dashboard, names='value')
dropdown_line.observe(update_dashboard, names='value')
slider_top_n.observe(update_dashboard, names='value')
date_picker_start.observe(update_dashboard, names='value')
date_picker_end.observe(update_dashboard, names='value')

# Exibir Dashboard
controls = widgets.VBox([widgets.HBox([date_picker_start, date_picker_end]), 
                         widgets.HBox([dropdown_country, dropdown_line, slider_top_n])])
display(controls, output_plot)

Output()

In [8]:
update_dashboard(None)